In [1]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [2]:
# Install required packages
!uv pip install langchain langchain-openai langchain-core python-dotenv

Using Python 3.12.13 environment at: /usr
Resolved 42 packages in 815ms
Prepared 3 packages in 87ms
Uninstalled 1 package in 16ms
Installed 3 packages in 12ms
 - langchain-core==1.3.1
 + langchain-core==1.3.3
 + langchain-openai==1.2.1
 + langchain-protocol==0.0.15


In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate

import os
from dotenv import load_dotenv

load_dotenv("/content/drive/MyDrive/Datasets/API_KEYS.env")
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

## Query Rewriting:Reformulating queries to improve retrieval

In [4]:
re_write_llm = ChatOpenAI(temperature=0, model_name="gpt-4o", max_tokens=4000)

# Create a prompt template for query rewriting
query_rewrite_template = """You are an AI assistant tasked with reformulating user queries to improve retrieval in a RAG system.
Given the original query, rewrite it to be more specific, detailed, and likely to retrieve relevant information.

original query:{original_query}

Rewritten query:"""

query_rewrite_prompt=PromptTemplate(
    input_variables=["original_query"],
    template=query_rewrite_template
)

# Create an LLMChain for query rewriting
query_rewriter = query_rewrite_prompt | re_write_llm

def rewrite_query(original_query):
    """
    Rewrite the original query to improve retrieval.

    Args:
    original_query (str): The original user query

    Returns:
    str: The rewritten query
    """
    response=query_rewriter.invoke(original_query)
    return response.content

## Demonstrate on use case

In [5]:
# example query over the understanding of climate change dataset
original_query="What are the impacts of climate change on the environment?"
rewritten_query=rewrite_query(original_query)
print(f"Original query:{original_query}")
print(f"\nRewritten query:{rewritten_query}")

Original query:What are the impacts of climate change on the environment?

Rewritten query:How does climate change affect various aspects of the environment, such as biodiversity, ecosystems, weather patterns, and sea levels?


## Step-back prompting: Generating broader queries for better context retrieval

In [6]:
step_back_llm = ChatOpenAI(temperature=0, model_name='gpt-4o', max_tokens=4000)

# Create a prompt template for step-back prompting
step_back_template = """You are an AI assistant tasked with generating broader, more general queries to improve context retrieval in a RAG system.
Given the original query, generate a step-back query that is more general and can help retrieve relevant background information.

Original query:{original_query}

step-back query:"""

step_back_prompt=PromptTemplate(
    input_variables=['original_query'],
    template=step_back_template
)

# Create an LLMChain for step-back prompting
step_back_chain = step_back_prompt | step_back_llm

def generate_step_back_query(original_query):
    """
    Generate a step-back query to retrieve broader context.

    Args:
    original_query (str): The original user query

    Returns:
    str: The step-back query
    """
    response = step_back_chain.invoke(original_query)
    return response.content

## Demonstrate on use case

In [7]:
# example query over the understanding climate change dataset
original_query="What are the impacts of climate change in the environment?"
step_back_query=generate_step_back_query(original_query)
print("Original query:", original_query)
print("\nStep-back query:", step_back_query)

Original query: What are the impacts of climate change in the environment?

Step-back query: What are the general effects of environmental changes on ecosystems and biodiversity?


## Sub-query decomposition: Breaking complex queries into simpler sub-queries

In [8]:
sub_query_llm = ChatOpenAI(temperature=0, model_name='gpt-4o', max_tokens=4000)

# Create a prompt template for sub-query decomposition
subquery_decomposition_template = """You are an AI assistant tasked with breaking down complex queries into simpler sub-queries for a RAG system.
Given the original query, decompose it into 2-4 simpler sub-queries that, when answered together, would provide a comprehensive response to the original query.

Original query: {original_query}

example: What are the impacts of climate change on the environment?

Sub-queries:
1. What are the impacts of climate change on biodiversity?
2. How does climate change affect the oceans?
3. What are the effects of climate change on agriculture?
4. What are the impacts of climate change on human health?"""

subquery_decomposition_prompt=PromptTemplate(
    input_variables=['original_query'],
    template=subquery_decomposition_template
)

# Create an LLMChain for sub-query decomposition
subquery_decomposer_chain=subquery_decomposition_prompt | sub_query_llm

def decompose_query(original_query:str):
    """
    Decompose the original query into simpler sub-queries.

    Args:
    original_query (str): The original complex query

    Returns:
    List[str]: A list of simpler sub-queries
    """
    response=subquery_decomposer_chain.invoke(original_query).content
    sub_queries=[q.strip() for q in response.split('\n') if q.strip() and not q.strip().startswith('Sub-queries')]
    return sub_queries

## Demonstration on a use case

In [9]:
# example query over the understanding climate change dataset
original_query="What are the impacts of climate change in the environment?"
sub_queries=decompose_query(original_query)
print("\nSub-queries:")
for i, sub_query in enumerate(sub_queries, 1):
  print(sub_query)


Sub-queries:
1. How does climate change affect weather patterns and extreme weather events?
2. What are the impacts of climate change on ecosystems and wildlife?
3. How does climate change influence sea levels and coastal areas?
4. What are the effects of climate change on freshwater resources and availability?


In [10]:
import os
from openai import OpenAI
from typing import List, Dict
import json
from IPython.display import display, Markdown, HTML

from dotenv import load_dotenv
load_dotenv("/content/drive/MyDrive/Datasets/API_KEYS.env")

client=OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("✅ Setup complete!")

✅ Setup complete!


# Multi-Query Generation

In [11]:
def multi_query_generation(original_query:str, num_queries:int=3)->List[str]:
      prompt = f"""You are an AI assistant helping to improve search queries.

Generate {num_queries} different versions of the following question.
Each version should:
- Ask the same thing but from a different perspective
- Use different wording and phrasing
- Help retrieve relevant information from a vector database

Original question: {original_query}

Return ONLY a JSON array of {num_queries} alternative questions, nothing else.
Example format: ["question 1", "question 2", "question 3"]"""
      response=client.chat.completions.create(
          model="gpt-4",
          messages=[{"role":"user", "content":prompt}],
          temperature=0.7
      )

      queries=json.loads(response.choices[0].message.content)
      return queries

In [12]:
original_query="What are the benefits of using RAG in LLM applications?"

print(f"📝 Original query:{original_query}\n")
variations=multi_query_generation(original_query, num_queries=3)

print("🔄 Generated variations:")
for i, var in enumerate(variations, 1):
  print(f" {i}.{var}")

📝 Original query:What are the benefits of using RAG in LLM applications?

🔄 Generated variations:
 1.What advantages does employing RAG bring to LLM applications?
 2.How does the use of RAG enhance LLM applications?
 3.In what ways can using RAG be beneficial for LLM applications?


## Query Routing

In [13]:
def query_routing(query:str, available_sources:List[Dict[str, str]]) -> Dict:
  sources_text = "\n".join([
      f"{i+1}.{source['name']}:{source['description']}"
      for i, source in enumerate(available_sources)
  ])
  prompt = f"""You are a query router. Given a user question and available data sources,
select the MOST appropriate source(s) to answer the question.

Available Sources:
{sources_text}

User Question: {query}

Return your answer as JSON with this structure:
{{
    "selected_sources": ["source_name1", "source_name2"],
    "reasoning": "brief explanation of why these sources"
}}"""
  response=client.chat.completions.create(
      model="gpt-4o",
      messages=[{'role':"user", "content":prompt}],
      temperature=0.3,
      response_format={"type":"json_object"}
  )
  result=json.loads(response.choices[0].message.content)
  return result

In [14]:
# Define available data sources
sources = [
    {"name": "ml_docs", "description": "Machine learning tutorials and best practices"},
    {"name": "deployment_guide", "description": "Cloud deployment and DevOps documentation"},
    {"name": "api_reference", "description": "API specifications and code examples"},
    {"name": "case_studies", "description": "Real-world implementation case studies"}
]

query="How do I deploy a machine learning model to production?"
print(f"📝 Query:{query}\n")
print(f"🗂️ Available Sources:{[s['name'] for s in sources]}\n")

routing_result=query_routing(query, sources)
print("🎯  Routing Decision:")
print(f"   Selected:{routing_result['selected_sources']}")
print(f"   Reasoning:{routing_result['reasoning']}")

📝 Query:How do I deploy a machine learning model to production?

🗂️ Available Sources:['ml_docs', 'deployment_guide', 'api_reference', 'case_studies']

🎯  Routing Decision:
   Selected:['deployment_guide', 'ml_docs']
   Reasoning:The 'deployment_guide' is the most relevant source as it contains documentation on cloud deployment and DevOps, which are essential for deploying a machine learning model to production. Additionally, 'ml_docs' may provide useful information on best practices for deploying machine learning models.


## Query Compression

In [15]:
def query_compression(query: str, conversation_history: List[Dict[str, str]] = None) -> str:
    context = ""
    if conversation_history:
        context = "\n".join([
            f"{msg['role']}: {msg['content']}"
            for msg in conversation_history[-3:]
        ])
        context = f"\nConversation History:\n{context}\n"

    prompt = f"""You are a query optimization expert. Compress the following query into a concise search query.

{context}
Current Query: {query}

Rules:
- Remove filler words and unnecessary context
- Keep only the core semantic concepts
- Maintain the search intent
- Output should be 3-8 words maximum
- Focus on keywords that would match relevant documents

Return ONLY the compressed query, nothing else."""

    response = client.chat.completions.create(
        model="gpt-4",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )

    compressed_query = response.choices[0].message.content.strip()
    return compressed_query

In [16]:
verbose_query="I was wondering if you could help me understand what the main advantages are of implementing a Retrieval Augmented Generation system compared to just using a standard language model?"
print(f"📝 Verbose Query ({len(verbose_query)} chars):\n{verbose_query}\n")

compressed = query_compression(verbose_query)
print(f"✂️ Compressed Query ({len(compressed)} chars):\n{compressed}\n")
print(f"Compression Ratio:{len(compressed)/len(verbose_query)*100:.1f}%")

📝 Verbose Query (182 chars):
I was wondering if you could help me understand what the main advantages are of implementing a Retrieval Augmented Generation system compared to just using a standard language model?

✂️ Compressed Query (73 chars):
"Advantages of Retrieval Augmented Generation vs standard language model"

Compression Ratio:40.1%


## Compression with conversation history

In [17]:
# Test with conversation context
conversation = [
    {"role": "user", "content": "Tell me about systems"},
    {"role": "assistant", "content": " systems combine retrieval with generation..."}
]

query_with_context="What about the implementation details?"

compressed_with_context=query_compression(query_with_context, conversation)
print(f"Original:{query_with_context}")
print(f"Compressed:{compressed_with_context}")

Original:What about the implementation details?
Compressed:"Systems implementation details"


## Contextual Query Enhancement

In [18]:
def contextual_query_enhancement(
    query:str,
    user_context:Dict[str, str],
    conversation_history:List[Dict[str, str]]=None
) -> str:
  context_text="\n".join([f"- {k}:{v}" for k, v in user_context.items()])

  history_text=""
  if conversation_history:
    history_text="\n".join([
        f"{msg['role']}:{msg['content']}"
        for msg in conversation_history[-3:]
    ])
  prompt = f"""You are enhancing a search query with contextual information.

User Context:
{context_text}

{f"Recent Conversation:{chr(10)}{history_text}" if history_text else ""}

Original Query: {query}

Task: Enhance this query by:
1. Adding relevant context from user profile
2. Incorporating conversation history if relevant
3. Making implicit information explicit
4. Keeping it natural and search-friendly

Return ONLY the enhanced query, nothing else."""
  response=client.chat.completions.create(
      model="gpt-4",
      messages=[{"role":"user", "content":prompt}],
      temperature=0.5
  )

  enhanced_query=response.choices[0].message.content.strip()
  return enhanced_query

In [19]:
query="What are the best practices?"
user_context = {
    "role": "Senior Data Engineer",
    "experience": "5 years",
    "focus_area": "MLOps and model deployment",
    "current_stack": "Python, Docker, Kubernetes"
}

conv_history = [
    {"role": "user", "content": "Tell me about deploying ML models"},
    {"role": "assistant", "content": "There are several approaches to model deployment..."}
]

print(f"📝 Original Query:{query}\n")
print(f"👤 User Context:{user_context}\n")

enhanced=contextual_query_enhancement(query, user_context, conv_history)

print(f"✨ Enhanced Query:{enhanced}")

📝 Original Query:What are the best practices?

👤 User Context:{'role': 'Senior Data Engineer', 'experience': '5 years', 'focus_area': 'MLOps and model deployment', 'current_stack': 'Python, Docker, Kubernetes'}

✨ Enhanced Query:What are the best practices for deploying ML models in a MLOps environment using Python, Docker, and Kubernetes with 5 years of experience as a Senior Data Engineer?


### Personalize for different users

In [20]:
profiles = [
    {"role": "Data Scientist", "experience": "2 years", "focus_area": "Deep Learning"},
    {"role": "ML Engineer", "experience": "7 years", "focus_area": "Production Systems"},
    {"role": "Research Scientist", "experience": "10 years", "focus_area": "NLP Research"}
]

query="How do I optimize model performance?"

for profile in profiles:
  enhanced=contextual_query_enhancement(query, profile)
  print(f"\n{profile['role']}:{enhanced}")


Data Scientist:"How do I optimize deep learning model performance with 2 years of data science experience?"

ML Engineer:What are advanced strategies to optimize machine learning model performance for production systems with 7 years of experience as an ML Engineer?

Research Scientist:What are the best practices for optimizing performance of NLP models for a research scientist with 10 years of experience?


# Query Translation

In [23]:
def query_translation_to_filters(query:str, available_filters:Dict[str, List]) -> Dict:
  filters_description="\n".join([
      f"-{field}:{','.join(map(str, values))}"
      for field, values in available_filters.items()
  ])
  prompt = f"""You are a query translator. Extract the semantic query and metadata filters from a natural language query.

Available Metadata Filters:
{filters_description}

Natural Language Query: {query}

Parse this into:
1. A semantic search query (the conceptual question)
2. Metadata filters (specific attributes to filter on)

Return JSON in this format:
{{
    "semantic_query": "the core question without filter terms",
    "filters": {{
        "field_name": "value or condition"
    }}
}}

Only include filters that are explicitly mentioned in the query."""
  response=client.chat.completions.create(
      model="gpt-4o",
      messages=[{"role":"user", "content":prompt}],
      temperature=0.2,
      response_format={"type":"json_object"}
  )
  result=json.loads(response.choices[0].message.content)
  return result

In [24]:
query="Show me research papers about transformers published after 2020with high citations"
available_filters = {
    "year": list(range(2018, 2025)),
    "topic": ["transformers", "CNN", "RNN", "attention", "RAG"],
    "citation_count": ["low", "medium", "high"],
    "author": ["various"]
}
print(f"📝 Natural Language Query:\n{query}\n")
print(f"🔧 Available Filters:\n{json.dumps(available_filters, indent=2)}\n")

translation=query_translation_to_filters(query, available_filters)
print("🎯 Translation Result:")
print(json.dumps(translation, indent=2))

📝 Natural Language Query:
Show me research papers about transformers published after 2020with high citations

🔧 Available Filters:
{
  "year": [
    2018,
    2019,
    2020,
    2021,
    2022,
    2023,
    2024
  ],
  "topic": [
    "transformers",
    "CNN",
    "RNN",
    "attention",
    "RAG"
  ],
  "citation_count": [
    "low",
    "medium",
    "high"
  ],
  "author": [
    "various"
  ]
}

🎯 Translation Result:
{
  "semantic_query": "research papers about transformers",
  "filters": {
    "year": ">2020",
    "citation_count": "high"
  }
}


## E-commerce Search

In [26]:
ecommerce_filters = {
    "category": ["electronics", "clothing", "home", "sports"],
    "price_range": ["under_50", "50-100", "100-500", "over_500"],
    "brand": ["Apple", "Samsung", "Nike", "various"],
    "rating": ["1-2", "3", "4", "5"],
    "in_stock": ["yes", "no"]
}

ecommerce_query="Find me highly rated Apple laptops under $1000 that are in stock"

result=query_translation_to_filters(ecommerce_query, ecommerce_filters)
print(json.dumps(result, indent=2))

{
  "semantic_query": "Find Apple laptops",
  "filters": {
    "rating": "5",
    "brand": "Apple",
    "price_range": "100-500",
    "in_stock": "yes"
  }
}
